# 02 — Test Model Module
Детальный анализ результатов 3-Statement модели.

**Разделы:**
1. Настройка и запуск модели
2. Валидация: BS Identity, CF Bridge, RE Bridge
3. IS / BS / CF детальный анализ
4. Corkscrew анализ (PPE, WC, Debt, NOL, Lease)
5. Стресс-тестирование (из build_model)
6. Рейтинг и ковенанты (из build_model)
7. Итоговая сводка


## §1 Настройка

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR.parents[2]   if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

COMPANY_ID    = 'us_steel'
SCENARIO_NAME = 'base'
DB_PATH       = ROOT / 'data_mart_v2.db'
CONFIG_PATH   = ROOT / 'companies' / COMPANY_ID / 'configs' / 'project.yaml'

print(f'ROOT:   {ROOT}')
print(f'DB:     {DB_PATH.name} exists={DB_PATH.exists()}')
print(f'Config: {CONFIG_PATH.name} exists={CONFIG_PATH.exists()}')

In [ ]:
import logging
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s — %(message)s')

from engine.orchestrator import build_model

result = build_model(
    company_id=COMPANY_ID,
    scenario_name=SCENARIO_NAME,
    db_path=DB_PATH,
    config_path=CONFIG_PATH,
    run_preprocessor=False,
    run_macro=False,
    run_model=True,
    run_stress=True,
    run_rating=True,
    run_covenants=True,
)

mr = result.model_result
print('Статус:', 'OK OK' if result.success else 'FAIL FAIL')
if result.success:
    print(f'BS max diff: {max(mr.bs_diffs.values()):.6f}')
    print(f'CF max diff: {max(mr.cf_diffs.values()):.6f}')
    print(f'Лет прогноза: {len(mr.years)}')
    print(f'Стресс-сценарии: {list(result.stress_results.keys())}')
if not result.success:
    print('Errors:', result.errors)


## §2 Валидация: BS Identity / CF Bridge / RE Bridge

In [ ]:
# BS Identity: Total Assets = Total Liabilities + Total Equity
print('BS Identity (Assets = Liab + Equity):')
print(f'  {"Год":<6} {"Assets$B":>9} {"L+E$B":>9} {"Diff":>10} {"OK?"}')
for yr, s in sorted(mr.years.items()):
    ok, msg, diff = s.bs_check()
    icon = '✅' if abs(diff) < 1 else '❌'
    print(f'  {yr:<6} {s.total_assets/1e9:>9.3f} {s.total_liab_equity/1e9:>9.3f} {diff:>10.2f} {icon}')


In [ ]:
# CF Bridge: Cash_open + Net_change = Cash_end
print('CF Bridge (cash_open + net_change = cash_end):')
print(f'  {"Год":<6} {"Open$M":>8} {"NetChg$M":>10} {"End$M":>8} {"Diff":>8} {"OK?"}')
for yr, s in sorted(mr.years.items()):
    ok, msg, diff = s.cf_bridge_check()
    icon = '✅' if abs(diff) < 1 else '❌'
    print(f'  {yr:<6} {s.cf_cash_opening/1e6:>8.0f} {s.cf_net_change/1e6:>10.0f} {s.cf_cash_ending/1e6:>8.0f} {diff:>8.2f} {icon}')


In [ ]:
# RE Bridge: RE_open + NI - Dividends = RE_close
print('RE Bridge:')
print(f'  {"Год":<6} {"RE_open":>9} {"NI":>7} {"Div":>7} {"RE_close":>10} {"Diff":>8}')
years = sorted(mr.years.keys())
for i, yr in enumerate(years):
    s = mr.years[yr]
    prev_re = mr.years[years[i-1]].retained_earnings if i > 0 else None
    if prev_re is not None:
        expected_re = prev_re + s.net_income + s.cff_dividends
        diff = s.retained_earnings - expected_re
        icon = '✅' if abs(diff) < 1e6 else '⚠️'
        print(f'  {yr:<6} {prev_re/1e6:>9.0f} {s.net_income/1e6:>7.0f} {s.cff_dividends/1e6:>7.0f} {s.retained_earnings/1e6:>10.0f} {diff/1e6:>8.1f}M {icon}')


## §3 Income Statement

In [ ]:
years = sorted(mr.years.keys())

IS_ITEMS = [
    ('Revenue',              lambda s: s.revenue,               False),
    ('COGS',                 lambda s: s.cogs,                  False),
    ('Gross Profit',         lambda s: s.gross_profit,          False),
    ('  gross margin %',     lambda s: s.gross_profit/s.revenue*100, True),
    ('SGA',                  lambda s: s.sga,                   False),
    ('Earnings investees',   lambda s: s.earnings_from_investees, False),
    ('EBITDA',               lambda s: s.ebitda,                False),
    ('  EBITDA margin %',    lambda s: s.ebitda/s.revenue*100,  True),
    ('D&A',                  lambda s: -s.total_da,             False),
    ('EBIT',                 lambda s: s.ebit,                  False),
    ('  EBIT margin %',      lambda s: s.ebit/s.revenue*100,    True),
    ('Interest expense',     lambda s: -s.interest_expense,     False),
    ('Interest income',      lambda s: s.interest_income,       False),
    ('Net periodic benefit', lambda s: s.net_periodic_benefit,  False),
    ('EBT',                  lambda s: s.ebt,                   False),
    ('Tax expense',          lambda s: s.tax_expense,           False),
    ('  Tax rate eff %',     lambda s: abs(s.tax_expense)/max(s.ebt,1)*100 if s.ebt > 0 else 0, True),
    ('Net Income',           lambda s: s.net_income,            False),
    ('  NI margin %',        lambda s: s.net_income/s.revenue*100, True),
]

rows = []
for name, fn, is_pct in IS_ITEMS:
    row = {'Статья': name}
    for yr in years:
        v = fn(mr.years[yr])
        row[yr] = f'{v:.1f}%' if is_pct else f'{v/1e6:.0f}'
    rows.append(row)

df_is = pd.DataFrame(rows).set_index('Статья')
print('Income Statement ($M)')
display(df_is)


## §4 Balance Sheet

In [ ]:
BS_ITEMS = [
    ('── ASSETS ──────────────────', lambda s: None),
    ('Cash',                    lambda s: s.cash),
    ('Accounts Receivable',     lambda s: s.accounts_receivable),
    ('Inventory',               lambda s: s.inventory),
    ('Other CA',                lambda s: s.other_ca),
    ('Total Current Assets',    lambda s: s.total_ca),
    ('PPE gross',               lambda s: s.ppe_gross),
    ('PPE accum dep',           lambda s: -s.ppe_accum_dep),
    ('PPE net',                 lambda s: s.ppe_net),
    ('ROU Asset',               lambda s: s.rou_asset),
    ('Intangibles',             lambda s: s.intangibles),
    ('Goodwill',                lambda s: s.goodwill),
    ('DTA',                     lambda s: s.dta),
    ('Other NCA',               lambda s: s.other_nca),
    ('Total NCA',               lambda s: s.total_nca),
    ('TOTAL ASSETS',            lambda s: s.total_assets),
    ('── LIABILITIES ─────────────', lambda s: None),
    ('Short-term Debt',         lambda s: s.short_term_debt),
    ('Accounts Payable',        lambda s: s.accounts_payable),
    ('Taxes Payable',           lambda s: s.taxes_payable),
    ('Interest Payable',        lambda s: s.interest_payable),
    ('Lease Liab Current',      lambda s: s.lease_liab_current),
    ('Other CL',                lambda s: s.other_cl),
    ('Total Current Liab',      lambda s: s.total_cl),
    ('Long-term Debt',          lambda s: s.long_term_debt),
    ('Employee Benefits',       lambda s: s.employee_benefits),
    ('Lease Liab NCurrent',     lambda s: s.lease_liab_noncurrent),
    ('DTL',                     lambda s: s.dtl),
    ('Other NCL',               lambda s: s.other_ncl),
    ('Total NCL',               lambda s: s.total_ncl),
    ('Total Liabilities',       lambda s: s.total_liabilities),
    ('── EQUITY ──────────────────', lambda s: None),
    ('Share Capital',           lambda s: s.share_capital),
    ('APIC',                    lambda s: s.apic),
    ('Retained Earnings',       lambda s: s.retained_earnings),
    ('Treasury Stock',          lambda s: s.treasury_stock),
    ('AOCI',                    lambda s: s.aoci),
    ('NCI',                     lambda s: s.nci),
    ('Total Equity',            lambda s: s.total_equity),
    ('TOTAL L+E',               lambda s: s.total_liab_equity),
    ('✓ BS diff',               lambda s: s.bs_check()[2]),
]

rows = []
for name, fn in BS_ITEMS:
    row = {'Статья': name}
    for yr in years:
        v = fn(mr.years[yr])
        if v is None:
            row[yr] = ''
        elif '✓' in name:
            row[yr] = f'{v:.2f}'
        elif '──' in name:
            row[yr] = ''
        else:
            row[yr] = f'{v/1e6:.0f}'
    rows.append(row)

df_bs = pd.DataFrame(rows).set_index('Статья')
print('Balance Sheet ($M)')
display(df_bs)


## §5 Cash Flow Statement

In [ ]:
CF_ITEMS = [
    ('── CFO ──────────────────────', lambda s: None),
    ('Net Income',               lambda s: s.cfo_net_income),
    ('D&A',                      lambda s: s.cfo_total_da),
    ('Deferred Tax',             lambda s: s.cfo_deferred_tax),
    ('Δ AR',                     lambda s: s.cfo_change_ar),
    ('Δ Inventory',              lambda s: s.cfo_change_inv),
    ('Δ AP',                     lambda s: s.cfo_change_ap),
    ('WC Delta total',           lambda s: s.cfo_wc_delta),
    ('Interest paid',            lambda s: -s.cfo_interest_paid),
    ('Taxes paid',               lambda s: -s.cfo_taxes_paid),
    ('Lease payments (op)',      lambda s: s.cfo_lease_payments_operating),
    ('CFO Total',                lambda s: s.cfo_total),
    ('── CFI ──────────────────────', lambda s: None),
    ('CapEx',                    lambda s: s.cfi_capex),
    ('Disposal proceeds',        lambda s: s.cfi_disposal_proceeds),
    ('CFI Total',                lambda s: s.cfi_total),
    ('── CFF ──────────────────────', lambda s: None),
    ('Debt issuance',            lambda s: s.cff_debt_issuance),
    ('Debt repayment',           lambda s: s.cff_debt_repayment),
    ('Dividends',                lambda s: s.cff_dividends),
    ('Finance lease principal',  lambda s: s.cff_finance_lease_principal),
    ('CFF Total',                lambda s: s.cff_total),
    ('── BRIDGE ───────────────────', lambda s: None),
    ('Net Change',               lambda s: s.cf_net_change),
    ('Cash Opening',             lambda s: s.cf_cash_opening),
    ('Cash Ending',              lambda s: s.cf_cash_ending),
    ('✓ CF diff',                lambda s: s.cf_bridge_check()[2]),
]

rows = []
for name, fn in CF_ITEMS:
    row = {'Статья': name}
    for yr in years:
        v = fn(mr.years[yr])
        if v is None or '──' in name:
            row[yr] = ''
        elif '✓' in name:
            row[yr] = f'{v:.2f}'
        else:
            row[yr] = f'{v/1e6:.0f}'
    rows.append(row)

df_cf = pd.DataFrame(rows).set_index('Статья')
print('Cash Flow Statement ($M)')
display(df_cf)


## §6 Ключевые коэффициенты

In [ ]:
from engine.database.repository import Repository
from engine.model.loader import ModelInputLoader

with Repository(db_path=DB_PATH) as repo:
    loader = ModelInputLoader(COMPANY_ID, repo, CONFIG_PATH, SCENARIO_NAME)
    historic, config = loader.load()

base = historic.base_year_state

KPI_ITEMS = [
    ('Revenue growth %',       lambda s,p: (s.revenue/p.revenue-1)*100 if p else None),
    ('EBITDA margin %',        lambda s,p: s.ebitda/s.revenue*100),
    ('EBIT margin %',          lambda s,p: s.ebit/s.revenue*100),
    ('NI margin %',            lambda s,p: s.net_income/s.revenue*100),
    ('D&A / Revenue %',        lambda s,p: s.total_da/s.revenue*100),
    ('CapEx / Revenue %',      lambda s,p: abs(s.cfi_capex)/s.revenue*100),
    ('FCF $M',                 lambda s,p: (s.cfo_total+s.cfi_capex)/1e6),
    ('Net Debt $B',            lambda s,p: (s.short_term_debt+s.long_term_debt-s.cash)/1e9),
    ('Net Debt / EBITDA x',    lambda s,p: (s.short_term_debt+s.long_term_debt-s.cash)/s.ebitda if s.ebitda>0 else None),
    ('Interest coverage x',    lambda s,p: s.ebit/s.interest_expense if s.interest_expense>0 else None),
    ('ST debt / Total debt %', lambda s,p: s.short_term_debt/(s.short_term_debt+s.long_term_debt)*100 if (s.short_term_debt+s.long_term_debt)>0 else 0),
    ('FCF / ST debt x',        lambda s,p: (s.cfo_total+s.cfi_capex)/s.short_term_debt if s.short_term_debt>0 else None),
    ('Current ratio x',        lambda s,p: s.total_ca/s.total_cl if s.total_cl>0 else None),
]

rows = []
for name, fn in KPI_ITEMS:
    row = {'KPI': name}
    for i, yr in enumerate(years):
        s = mr.years[yr]
        p = mr.years[years[i-1]] if i > 0 else base
        v = fn(s, p)
        row[yr] = f'{v:.1f}' if v is not None else 'N/A'
    rows.append(row)

df_kpi = pd.DataFrame(rows).set_index('KPI')
display(df_kpi)


## §7 Corkscrew анализ

In [ ]:
print('PPE Corkscrew ($M):')
print(f'  {"Год":<6} {"Gross_open":>11} {"CapEx":>7} {"Dep":>7} {"Gross_close":>12} {"Accum":>8} {"Net":>8}')
prev_gross = base.ppe_gross
for yr in years:
    s = mr.years[yr]
    dep = s.dep_ppe
    capex = abs(s.cfi_capex)
    print(f'  {yr:<6} {prev_gross/1e6:>11.0f} {capex/1e6:>7.0f} {dep/1e6:>7.0f} {s.ppe_gross/1e6:>12.0f} {s.ppe_accum_dep/1e6:>8.0f} {s.ppe_net/1e6:>8.0f}')
    prev_gross = s.ppe_gross


In [ ]:
print('WC Corkscrew ($M):')
print(f'  {"Год":<6} {"AR":>8} {"Inv":>8} {"AP":>8} {"NWC":>9} {"ΔNWC":>8} {"DSO":>6} {"DIH":>6} {"DPO":>6}')
prev_nwc = (base.accounts_receivable + base.inventory - abs(base.accounts_payable))
for yr in years:
    s = mr.years[yr]
    nwc = s.accounts_receivable + s.inventory - abs(s.accounts_payable)
    delta = nwc - prev_nwc
    dso = s.accounts_receivable / s.revenue * 365 if s.revenue > 0 else 0
    dih = s.inventory / abs(s.cogs) * 365 if s.cogs != 0 else 0
    dpo = abs(s.accounts_payable) / abs(s.cogs) * 365 if s.cogs != 0 else 0
    print(f'  {yr:<6} {s.accounts_receivable/1e6:>8.0f} {s.inventory/1e6:>8.0f} {s.accounts_payable/1e6:>8.0f} {nwc/1e6:>9.0f} {delta/1e6:>8.0f} {dso:>6.0f} {dih:>6.0f} {dpo:>6.0f}')
    prev_nwc = nwc


In [ ]:
print('Debt Corkscrew ($M):')
print(f'  {"Год":<6} {"ST":>7} {"LT":>8} {"Total":>8} {"Issue":>8} {"Repay":>8} {"IntExp":>8} {"FCF/ST":>8}')
for yr in years:
    s = mr.years[yr]
    total = s.short_term_debt + s.long_term_debt
    fcf = s.cfo_total + s.cfi_capex
    fcf_st = fcf/s.short_term_debt if s.short_term_debt > 0 else float('inf')
    fcf_st_str = f'{fcf_st:.1f}x' if fcf_st != float('inf') else 'inf'
    print(f'  {yr:<6} {s.short_term_debt/1e6:>7.0f} {s.long_term_debt/1e6:>8.0f} {total/1e6:>8.0f} {s.cff_debt_issuance/1e6:>8.0f} {s.cff_debt_repayment/1e6:>8.0f} {s.interest_expense/1e6:>8.0f} {fcf_st_str:>8}')


## §7.4 NOL — Использование переноса убытков
TCJA: max 80% taxable income, indefinite carryforward.

In [ ]:
# NOL carryforward utilization analysis
# tax_expense < 0 means tax expense (reduces NI); compute effective rate and shield
STATUTORY_RATE = 0.21

print('NOL utilization ($M):')
print(f'  {"Год":<6} {"EBT":>8} {"Stat(21%)": >10} {"ActualTax":>10} {"Shield":>8} {"Eff%":>7}')
print('  ' + '-'*58)
for yr in years:
    s = mr.years[yr]
    ebt    = s.ebt
    tax    = s.tax_expense           # negative
    stat   = -ebt * STATUTORY_RATE   # positive
    actual = -tax                    # positive outflow
    shield = stat - actual
    eff    = actual / ebt * 100 if ebt > 0 else 0.0
    print(f'  {yr:<6} {ebt/1e6:>8.0f} {stat/1e6:>10.0f} {actual/1e6:>10.0f} {shield/1e6:>8.0f} {eff:>7.1f}%')

s25 = mr.years[2025]
shield25 = -(s25.ebt * STATUTORY_RATE + s25.tax_expense)
print()
print(f'  NOL открытие: $1,014M (project.yaml: taxes.nol_opening_balance)')
print(f'  Налоговая экономия 2025: ${shield25/1e6:.0f}M')
print(f'  Ставка 2027+: {STATUTORY_RATE*100:.0f}% (NOL исчерпан)')


## §7.5 Lease Corkscrew — ASC 842 Operating
ROU open → amort → close; LL open → principal → close. BS identity: ΔROU = ΔLL.

In [ ]:
# Lease corkscrew: operating leases (ASC 842)
# Cash payment is already in SGA -> NI; no separate CF adjustments needed
from engine.database.repository import Repository
from engine.model.loader import ModelInputLoader

with Repository(db_path=DB_PATH) as repo:
    loader2 = ModelInputLoader(COMPANY_ID, repo, CONFIG_PATH, SCENARIO_NAME)
    historic2, _ = loader2.load()

base_yr  = historic2.base_year_state
prev_rou = getattr(base_yr, 'rou_asset', 0.0) or 0.0
prev_ll  = (abs(getattr(base_yr, 'lease_liab_current', 0.0) or 0.0) +
            abs(getattr(base_yr, 'lease_liab_noncurrent', 0.0) or 0.0))

print('Operating Lease Corkscrew ($M):')
print(f'  {"Год":<6} {"ROU_op":>8} {"Delta_ROU":>10} {"LL_tot":>8} {"Delta_LL":>9} {"CashOut":>8} {"Match":>6}')
print('  ' + '-'*65)

for yr in years:
    s       = mr.years[yr]
    rou_cls = (getattr(s, 'rou_operating', None) or getattr(s, 'rou_asset', 0.0) or 0.0)
    ll_cls  = ((s.lease_liab_cur_operating or 0.0) + (s.lease_liab_ncur_operating or 0.0))
    d_rou   = rou_cls - prev_rou
    d_ll    = ll_cls  - prev_ll
    cash    = abs(s.cfo_lease_payments_operating or 0.0)
    match   = 'OK' if abs(d_rou - d_ll) < 1.0 else f'DIFF({d_rou-d_ll:.1f})'
    print(f'  {yr:<6} {rou_cls/1e6:>8.2f} {d_rou/1e6:>10.2f} {ll_cls/1e6:>8.2f} {d_ll/1e6:>9.2f} {cash/1e6:>8.2f} {match:>6}')
    prev_rou = rou_cls
    prev_ll  = ll_cls

print()
print('  ΔROU = ΔLL: BS identity preserved (operating leases, ASC 842)')
print('  Cash out embedded in NI via SGA — no addback in CFO')


## §8 Стресс-тестирование

In [ ]:
# Используем stress_results из build_model (run_stress=True)
stress_results = result.stress_results  # Dict[str, StressResult]

print('Стресс-сценарии:', list(stress_results.keys()))
print()
print(f'  {"Сценарий":<25} {"Rev25 D%":>9} {"EBITDA D%":>10} {"NI D%":>8} {"BS":>6}')
print('  ' + '-'*63)
for name, sr in stress_results.items():
    if sr.success and 2025 in sr.comparison:
        c = sr.comparison[2025]
        bs_diff = c.get('bs_diff', 0)
        print(f'  {name:<25} {c["revenue_delta_pct"]:>+8.1f}% {c["ebitda_delta_pct"]:>+9.1f}%'
              f' {c["ni_delta_pct"]:>+7.1f}% {bs_diff:>6.2f}')
    else:
        print(f'  {name:<25}  FAIL: {sr.errors}')

# Covenant auto-trigger
if 'covenant_breach' in stress_results:
    sr = stress_results['covenant_breach']
    print()
    print('  [covenant_breach auto-triggered]:')
    if sr.success and 2025 in sr.comparison:
        c = sr.comparison[2025]
        print(f'    Rev={c["revenue_delta_pct"]:+.1f}%  EBITDA={c["ebitda_delta_pct"]:+.1f}%  NI={c["ni_delta_pct"]:+.1f}%')
else:
    print()
    print('  covenant_breach: не запущен (нарушений ковенантов нет)')


In [ ]:
# Детальный прогноз по сценариям
print(f'  {"":25} {"base":>10}', end='')
for name in stress_results:
    print(f' {name[:12]:>13}', end='')
print()
print('  ' + '-'*80)

metrics_stress = [
    ('Revenue 2025 $B',    lambda s: s.revenue/1e9),
    ('EBITDA% 2025',       lambda s: s.ebitda/s.revenue*100),
    ('NI 2025 $M',         lambda s: s.net_income/1e6),
    ('Net Debt $B',        lambda s: (s.short_term_debt+s.long_term_debt-s.cash)/1e9),
    ('ND/EBITDA 2025',     lambda s: (s.short_term_debt+s.long_term_debt-s.cash)/s.ebitda if s.ebitda>0 else None),
]

base_2025 = mr.years[2025]
for mname, fn in metrics_stress:
    base_v = fn(base_2025)
    base_str = f'{base_v:.1f}' if base_v is not None else 'N/A'
    print(f'  {mname:<25} {base_str:>10}', end='')
    for name, sr in stress_results.items():
        if sr.success and 2025 in sr.stress_values:
            sv = sr.stress_values[2025]
            class FakeState:
                pass
            fs = FakeState()
            for k, v in sv.items():
                setattr(fs, k, v)
            fs.revenue = sv.get('revenue', base_2025.revenue)
            fs.ebitda  = sv.get('ebitda',  base_2025.ebitda)
            fs.net_income = sv.get('net_income', base_2025.net_income)
            fs.short_term_debt = sv.get('net_debt', 0) + sv.get('cash', 0)
            fs.long_term_debt  = 0
            fs.cash = sv.get('cash', base_2025.cash)
            try:
                v2 = fn(fs)
                print(f' {v2:.1f}' if v2 is not None else ' N/A', end='')
            except:
                print(' N/A', end='')
        else:
            print(' N/A', end='')
    print()


## §9 Рейтинг и ковенанты

In [ ]:
# Rating from build_model result
forecast_r = result.rating_result

if forecast_r and forecast_r.success:
    print('Рейтинги прогноза:')
    print(f'  {"Год":<6} {"Рейтинг":>8} {"ND/EBITDA":>10} {"ICR":>6}')
    print('  ' + '-'*35)
    for yr in sorted(mr.years.keys()):
        s  = mr.years[yr]
        ri = forecast_r.ratings.get(yr, {})
        rating  = ri.get('rating', 'N/A') if ri else 'N/A'
        nd_ebit = (s.short_term_debt+s.long_term_debt-s.cash)/s.ebitda if s.ebitda else None
        icr     = s.ebit/abs(s.interest_expense) if s.interest_expense else None
        nd_s = f'{nd_ebit:.1f}x' if nd_ebit is not None else 'N/A'
        icr_s = f'{icr:.1f}x' if icr is not None else 'N/A'
        print(f'  {yr:<6} {rating:>8} {nd_s:>10} {icr_s:>6}')
    print()
    print(f'  Best: {forecast_r.best_rating}   Worst: {forecast_r.worst_rating}')
else:
    print('Rating: N/A')


In [ ]:
# Covenants from build_model + covenant auto-trigger demo
cov_result = result.covenants_result

if cov_result:
    print(cov_result.summary())
    print()
    print(f'  {"Год":<6} {"Ковенант":<25} {"Значение":>9} {"Лимит":>7} {"Запас%":>8} {"Статус":>6}')
    print('  ' + '-'*68)
    for r in cov_result.results:
        val_s = f'{r.value:.2f}' if r.value is not None else 'N/A'
        hr_s  = f'{r.headroom*100:+.1f}%'
        icon = 'OK' if r.status == 'ok' else 'WARN' if r.status == 'warning' else 'BREACH'
        print(f'  {r.year:<6} {r.covenant_name:<25} {val_s:>9} {r.threshold:>7.2f} {hr_s:>8} {icon:>6}')
    print()
    breaches = cov_result.breaches()
    if breaches:
        print(f'  COVENANT BREACH: {len(breaches)} нарушений')
        if 'covenant_breach' in result.stress_results:
            sc = result.stress_results['covenant_breach']
            if sc.success and sc.comparison:
                yr0 = min(sc.comparison.keys())
                c = sc.comparison[yr0]
                print(f'  covenant_breach stress ({yr0}): Rev={c["revenue_delta_pct"]:+.1f}% EBITDA={c["ebitda_delta_pct"]:+.1f}%')
    else:
        print('  Все ковенанты соблюдены — covenant_breach не авто-запускается')
        print('  (для теста авто-триггера создайте агрессивный сценарий с breach)')
else:
    print('Covenants: N/A (run_covenants=True required)')


## §10 Итоговая сводка

In [ ]:
print('=' * 60)
print(f'ИТОГОВАЯ СВОДКА: {COMPANY_ID.upper()} / {SCENARIO_NAME}')
print('=' * 60)
print()
print(f'  Модель: BS diff={max(mr.bs_diffs.values()):.2f}  CF diff={max(mr.cf_diffs.values()):.2f}')
print()

s25 = mr.years[2025]
s29 = mr.years[2029]
print(f'  {"Метрика":<25} {"2025":>10} {"2029":>10} {"D":>8}')
print('  ' + '-'*55)
items_summary = [
    ('Revenue $B',      s25.revenue/1e9,            s29.revenue/1e9),
    ('EBITDA $M',       s25.ebitda/1e6,             s29.ebitda/1e6),
    ('EBITDA margin %', s25.ebitda/s25.revenue*100, s29.ebitda/s29.revenue*100),
    ('NI $M',           s25.net_income/1e6,         s29.net_income/1e6),
    ('Tax rate %',     -s25.tax_expense/s25.ebt*100 if s25.ebt>0 else 0,
                       -s29.tax_expense/s29.ebt*100 if s29.ebt>0 else 0),
    ('FCF $M',          (s25.cfo_total+s25.cfi_capex)/1e6, (s29.cfo_total+s29.cfi_capex)/1e6),
    ('Net Debt $B',     (s25.short_term_debt+s25.long_term_debt-s25.cash)/1e9,
                        (s29.short_term_debt+s29.long_term_debt-s29.cash)/1e9),
]
for name, v25, v29 in items_summary:
    delta = v29 - v25
    print(f'  {name:<25} {v25:>10.1f} {v29:>10.1f} {delta:>+8.1f}')

print()
if result.rating_result and result.rating_result.success:
    r25 = result.rating_result.ratings.get(2025, {}).get('rating', 'N/A')
    r29 = result.rating_result.ratings.get(2029, {}).get('rating', 'N/A')
    print(f'  Рейтинг 2025: {r25}  |  Рейтинг 2029: {r29}')

if result.covenants_result:
    nb_br = len(result.covenants_result.breaches())
    fb    = result.covenants_result.first_breach_year()
    print(f'  Нарушений ковенантов: {nb_br}', end='')
    if fb:
        print(f'  (первое: {fb})', end='')
    print()

if result.stress_results:
    print()
    print('  Стресс (Rev D% 2025):')
    for sname, sr in result.stress_results.items():
        if sr.success and 2025 in sr.comparison:
            d = sr.comparison[2025].get('revenue_delta_pct', 0)
            print(f'    {sname:<25} {d:>+6.1f}%')
